# M12 · 課1 ｜ 你的第一個 AI 營運分析(動手筆記本)

> 在 **GitHub Codespaces** 線上打開這個 repo,從上往下**一格一格按 ▶ 執行**。看得到輸出就是成功。

**今天做出什麼**:讓三個 AI 角色(資料檢查員 → 分析洞察員 → 決策摘要員)讀你的營運資料,給出**固定四欄**的決策建議。配合投影片 U12 課1 一起看。

路線:① 看資料 → ② 認識三角色與格式藍圖 → ③ 跑一次 → ④ 換你試。

## 為什麼要這樣做?

一個人從第一筆看到最後一筆,看到後面就累、就會漏 —— 這不是你的問題,人就是這樣。
**怎麼辦呢?** 找三個角色接力:一個只挑錯、一個只分析、一個只給建議。錯誤就不會一路被帶下去。

先看看我們要分析的資料長什麼樣子 👇(按左邊的 ▶)

In [ ]:
import csv
rows = list(csv.reader(open('data/sales_sample.csv', encoding='utf-8')))
head, body = rows[0], rows[1:]
print('欄位:', head)
for r in body: print(r)
print(f'\n共 {len(body)} 筆。看得出哪幾筆怪怪的嗎?(提示:數量有 -5 和 300)')

## 先認識:三個角色 + 「格式藍圖」

- **資料檢查員**:只挑出怪怪、要人確認的地方(不分析)。
- **分析洞察員**:看趨勢與風險,而且每個結論都要有依據。
- **決策摘要員**:整理成老闆能直接做的建議。

重點:最後一定要交出**固定四欄**,這叫「格式藍圖」。跑下面這格看看藍圖規定了哪四欄 👇

In [ ]:
from src.blueprint import Decision
for name, f in Decision.model_fields.items():
    print(f'{name:13s} → {f.description}')

## 跑一次(這格需要金鑰)

按下面這格。**還沒設金鑰也沒關係** —— 它會提醒你怎麼設;設好 Codespaces Secrets 後**重跑這一格**,就會看到三個角色接力、最後印出四欄決策摘要。

> 金鑰只放 Codespaces Secrets,**不要**貼進程式碼或 commit。設定步驟看 `README.md` / 教師指南。

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()
data = Path('data/sales_sample.csv').read_text(encoding='utf-8')
try:
    from src.crew import build_crew
    result = build_crew(data).kickoff()
    print('\n===== 決策摘要 =====')
    print(result.pydantic.model_dump_json(indent=2) if result.pydantic else result.raw)
except RuntimeError as e:
    if 'NO_KEY' in str(e):
        print('⚠️ 還沒設金鑰。到 Codespaces Secrets 設一把(README / 教師指南有步驟),設好重跑這一格,就會看到三角色輸出。')
    else:
        raise

## 換你試 + 一句話帶走

- **換你試**:把上面 `data/sales_sample.csv` 某個品項的數量改成很誇張的數(例如 9999),重跑跑看 —— `anomalies` 會不會把它抓出來?
- **回家作業**:換一個你自己關心的營運問題,跑一次,記下一個它的限制。

> **一句話帶走:AI 給的每個結論,都要說得出依據;說不出,就別信。**

## (選配)把建議變成你「AI 小店」的台詞 🏪

跑完上面 live 那格(有金鑰)後,執行下面這格 → 它把你的 `action_items` 寫進 `shop-showcase/data.js`。
然後打開 `shop-showcase/`(在終端機 `cd ../shop-showcase && python3 -m http.server 8000`),**AI 店員講的就是你剛算出來的建議** —— 這就是你的成果展示,也能嵌進 M13 個人網站。

In [ ]:
import json, os
brand = '大甲鍋貼'  # ← 改成你的店名
try:
    items = list(result.pydantic.action_items) if result.pydantic else []
except Exception:
    items = []
if not items:
    items = ['先跑上面那格(需要金鑰)就會換成你真正算出來的建議']
os.makedirs('../shop-showcase', exist_ok=True)
js = 'window.SHOP_DATA = ' + json.dumps({'brand': brand, 'aiTips': items}, ensure_ascii=False) + ';'
open('../shop-showcase/data.js', 'w', encoding='utf-8').write(js)
print('✓ 已寫入 ../shop-showcase/data.js,打開 shop-showcase 就會看到 AI 店員講你的建議')
print(js)